# Dataset Exploration: Comprehensive Overview for TF-IDF + UGC Recommendation

This notebook evaluates all available datasets for content-based recommendation using TF-IDF + UGC:

- **archive_2**: Food.com recipes and interactions
- **archive_3**: Recipe reviews dataset
- **archive_4**: Yelp academic dataset (business, reviews, tips, users)
- **archive_6**: Food coding dataset
- **onlinefoods.csv**: Online food orders dataset
- **OpenFoodFacts**: Global food products database (TSV)

**Goal**: Evaluate datasets for TF-IDF + UGC content-based recommendation

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set data directory
DATA_DIR = Path('../data')

# Helper function for JSON lines
def load_json_lines(filepath, nrows=None):
    """Load JSON lines file efficiently"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if nrows and i >= nrows:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

print("✅ Environment setup complete!")

## 1. Archive_2: Food.com Dataset

In [ ]:
print("=" * 80)
print("ARCHIVE_2: Food.com Dataset")
print("=" * 80)

archive_2_dir = DATA_DIR / 'archive_2'
recipes_a2 = pd.read_csv(archive_2_dir / 'PP_recipes.csv')
interactions_train = pd.read_csv(archive_2_dir / 'interactions_train.csv')

print(f"\n📊 Recipes: {recipes_a2.shape} | Interactions: {interactions_train.shape}")
print(f"Columns: {list(recipes_a2.columns[:10])}...")
display(recipes_a2.head(2))

# Text columns for TF-IDF
text_cols_a2 = [col for col in recipes_a2.columns if any(kw in col.lower() 
                for kw in ['name', 'ingredient', 'step', 'tag', 'description'])]
print(f"\n🔤 Text columns: {text_cols_a2}")

## 2. Archive_3: Recipe Reviews Dataset

In [ ]:
print("=" * 80)
print("ARCHIVE_3: Recipe Reviews Dataset")
print("=" * 80)

archive_3_dir = DATA_DIR / 'archive_3'
recipes_a3 = pd.read_parquet(archive_3_dir / 'recipes.parquet') if (archive_3_dir / 'recipes.parquet').exists() else pd.read_csv(archive_3_dir / 'recipes.csv')
reviews_a3 = pd.read_parquet(archive_3_dir / 'reviews.parquet') if (archive_3_dir / 'reviews.parquet').exists() else pd.read_csv(archive_3_dir / 'reviews.csv')

print(f"\n📊 Recipes: {recipes_a3.shape} | Reviews: {reviews_a3.shape}")
display(recipes_a3.head(2))
display(reviews_a3.head(2))

# UGC analysis
review_text_cols = [col for col in reviews_a3.columns if 'text' in col.lower() or 'review' in col.lower()]
print(f"\n💬 UGC columns: {review_text_cols}")

## 3. Archive_4: Yelp Academic Dataset ⭐

In [ ]:
print("=" * 80)
print("ARCHIVE_4: Yelp Academic Dataset (JSON)")
print("=" * 80)
print("⚠️  Loading samples only (10k-50k rows) due to large file size\n")

archive_4_dir = DATA_DIR / 'archive_4'

# Load business data
business_a4 = load_json_lines(archive_4_dir / 'yelp_academic_dataset_business.json', nrows=10000)
print(f"\n📍 Business: {business_a4.shape}")
print(f"Columns: {list(business_a4.columns)}")
display(business_a4.head(2))

# Filter food-related
if 'categories' in business_a4.columns:
    food_businesses = business_a4[business_a4['categories'].notna() & 
                                   business_a4['categories'].str.contains('Restaurant|Food|Cafe', case=False, na=False)]
    print(f"\n🍽️  Food-related: {len(food_businesses)} / {len(business_a4)} ({len(food_businesses)/len(business_a4)*100:.1f}%)")

In [ ]:
# 餐厅评论表 (用户生成内容)
reviews_a4 = load_json_lines(archive_4_dir / 'yelp_academic_dataset_review.json', nrows=50000)
print(f"\n💬 Reviews (UGC): {reviews_a4.shape}")
display(reviews_a4.head(2))

if 'text' in reviews_a4.columns:
    review_lens = reviews_a4['text'].str.len()
    print(f"\n📝 Review text stats: Mean={review_lens.mean():.0f}, Median={review_lens.median():.0f} chars")
    print(f"Sample review:\n{reviews_a4['text'].iloc[0][:300]}...")

# Rating distribution
if 'stars' in reviews_a4.columns:
    plt.figure(figsize=(8, 4))
    reviews_a4['stars'].value_counts().sort_index().plot(kind='bar', color='gold')
    plt.title('Archive_4 (Yelp): Rating Distribution')
    plt.xlabel('Stars')
    plt.ylabel('Count')
    plt.show()

In [ ]:
# Load tips (short UGC)
tips_a4 = load_json_lines(archive_4_dir / 'yelp_academic_dataset_tip.json', nrows=20000)
print(f"\n💡 Tips (Short UGC): {tips_a4.shape}")
display(tips_a4.head(2))

# Load users
users_a4 = load_json_lines(archive_4_dir / 'yelp_academic_dataset_user.json', nrows=10000)
print(f"\n👥 Users: {users_a4.shape}")
if 'review_count' in users_a4.columns:
    print(f"Avg reviews per user: {users_a4['review_count'].mean():.1f}")

## 4. Archive_6: Food Coding Dataset

In [ ]:
print("=" * 80)
print("ARCHIVE_6: Food Coding Dataset")
print("=" * 80)

food_coded = pd.read_csv(DATA_DIR / 'archive_6' / 'food_coded.csv')
print(f"\n📊 Shape: {food_coded.shape}")
display(food_coded.head(2))
print(f"\nCategorical columns: {list(food_coded.select_dtypes(include='object').columns[:10])}")

## 5. Online Foods Dataset

In [ ]:
print("=" * 80)
print("ONLINE FOODS: Order Dataset")
print("=" * 80)

online_foods = pd.read_csv(DATA_DIR / 'onlinefoods.csv')
print(f"\n📊 Shape: {online_foods.shape}")
display(online_foods.head(2))
print(f"\nColumns: {list(online_foods.columns)}")

## 6. OpenFoodFacts Dataset

In [ ]:
print("=" * 80)
print("OPENFOODFACTS: Global Food Products Database")
print("=" * 80)
print("⚠️  Loading sample (50k rows) due to large file size\n")

openfood = pd.read_csv(DATA_DIR / 'en.openfoodfacts.org.products.tsv', 
                       sep='\t', nrows=50000, low_memory=False)

print(f"\n📊 Shape: {openfood.shape}")
print(f"Total columns: {len(openfood.columns)}")
display(openfood.head(2))

# Key text columns
key_text_cols = ['product_name', 'generic_name', 'categories', 'ingredients_text', 'brands']
available_text_cols = [col for col in key_text_cols if col in openfood.columns]
print(f"\n🔤 Text columns: {available_text_cols}")

for col in available_text_cols[:3]:
    if openfood[col].notna().any():
        print(f"\n{col}: {openfood[col].notna().sum()} non-null ({openfood[col].notna().sum()/len(openfood)*100:.1f}%)")
        print(f"Sample: {openfood[col].dropna().iloc[0][:100]}...")

## 7. 🎯 TF-IDF + UGC Suitability Assessment

In [ ]:
print("=" * 80)
print("📋 TF-IDF + UGC SUITABILITY ASSESSMENT FOR CONTENT-BASED RECOMMENDATION")
print("=" * 80)

assessment = pd.DataFrame([
    {
        'Dataset': 'Archive_2\n(Food.com)',
        'Content Features': 'Recipe name, ingredients,\ntags, steps',
        'UGC Available': 'Yes\n(ratings, reviews)',
        'Sample Size': f'{len(recipes_a2):,} recipes\n{len(interactions_train):,} interactions',
        'Text Quality': 'High\n(structured)',
        'TF-IDF Score': '⭐⭐⭐⭐⭐',
        'Recommendation': '✅ Excellent for\nrecipe reco'
    },
    {
        'Dataset': 'Archive_3\n(Recipe Reviews)',
        'Content Features': 'Recipe name, ingredients,\ndirections, tags',
        'UGC Available': 'Yes\n(detailed reviews)',
        'Sample Size': f'{len(recipes_a3):,} recipes\n{len(reviews_a3):,} reviews',
        'Text Quality': 'Very High\n(rich reviews)',
        'TF-IDF Score': '⭐⭐⭐⭐⭐',
        'Recommendation': '✅ Excellent for\nhybrid reco'
    },
    {
        'Dataset': 'Archive_4\n(Yelp) 🏆',
        'Content Features': 'Business name, categories,\nattributes',
        'UGC Available': f'YES!\n{len(reviews_a4):,} reviews\n{len(tips_a4):,} tips',
        'Sample Size': f'{len(business_a4):,} businesses\n(sample)',
        'Text Quality': 'Very High\n(detailed UGC)',
        'TF-IDF Score': '⭐⭐⭐⭐⭐',
        'Recommendation': '✅ BEST for\nrestaurant reco'
    },
    {
        'Dataset': 'Archive_6\n(Food Coding)',
        'Content Features': 'Food preferences\n(categorical)',
        'UGC Available': 'No\n(survey data)',
        'Sample Size': f'{len(food_coded):,} responses',
        'Text Quality': 'Low\n(categorical)',
        'TF-IDF Score': '⭐⭐',
        'Recommendation': '⚠️  User profiling\nnot TF-IDF'
    },
    {
        'Dataset': 'Online Foods',
        'Content Features': 'Order data\n(limited text)',
        'UGC Available': 'Minimal',
        'Sample Size': f'{len(online_foods):,} orders',
        'Text Quality': 'Low\n(categorical)',
        'TF-IDF Score': '⭐⭐',
        'Recommendation': '⚠️  Behavior\nanalysis only'
    },
    {
        'Dataset': 'OpenFoodFacts',
        'Content Features': 'Product name, ingredients,\ncategories, brands',
        'UGC Available': 'No\n(product DB)',
        'Sample Size': f'{len(openfood):,}+ products\n(sample)',
        'Text Quality': 'High\n(detailed)',
        'TF-IDF Score': '⭐⭐⭐⭐',
        'Recommendation': '✅ Good for\nproduct reco'
    }
])

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
display(assessment)

In [ ]:
print("\n" + "=" * 80)
print("🏆 RECOMMENDED DATASETS FOR TF-IDF + UGC CONTENT-BASED RECOMMENDATION")
print("=" * 80)
print("""
TOP 3 CHOICES:

🥇 BEST: Archive_4 (Yelp Academic Dataset)
   ✓ Rich UGC: 50k+ reviews with detailed text
   ✓ Tips: Short-form user content
   ✓ Business metadata: categories, attributes
   ✓ Large scale: Can filter for food/restaurants
   ✓ Perfect for: TF-IDF on reviews + business features
   💡 Use Case: Restaurant recommendation with user sentiment

🥈 Archive_3 (Recipe Reviews)
   ✓ High-quality review text
   ✓ Structured recipe content (ingredients, directions)
   ✓ Good for: Hybrid (content + sentiment)
   💡 Use Case: Recipe recommendation with review analysis

🥉 Archive_2 (Food.com)
   ✓ Clean recipe data with ingredients/tags
   ✓ User interaction data (ratings)
   ✓ Good for: Pure content-based filtering
   💡 Use Case: Recipe similarity matching

BONUS:
   • OpenFoodFacts: Ingredient-based product recommendation
   • Archive_6 + Online Foods: User profiling & demographics

""")

print("=" * 80)
print("📊 IMPLEMENTATION PRIORITY")
print("=" * 80)
print("""
Phase 1: Archive_4 (Yelp)
  → TF-IDF on review text
  → Business category filtering
  → Content-based similarity

Phase 2: Archive_3 (Recipe Reviews)
  → TF-IDF on recipe content + reviews
  → Sentiment analysis on UGC
  → Hybrid recommendation

Phase 3: Archive_2 (Food.com)
  → TF-IDF on ingredients/tags
  → Collaborative filtering on interactions
  → Matrix factorization
""")